# Iris Classification — Reproducible Evaluation

Modernised version of the original internship Iris notebook. The original `IRIS_dataset_internship.ipynb` is preserved unchanged. This notebook uses **scikit-learn's bundled Iris dataset**, rather than the missing `/content/iris.csv`, and therefore its results are a new evaluation, not a reproduction of the original saved scores.

**Goal:** compare K-nearest neighbors (KNN) and support vector classification (SVC) on an untouched holdout set after cross-validation on training data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

RANDOM_STATE = 42

## Load and inspect the data

`load_iris()` provides 150 examples with four numeric flower measurements and three species. No local CSV is required.

In [ ]:
iris = load_iris(as_frame=True)
X = iris.data.copy()
y = iris.target.copy()
print('Shape:', X.shape)
print('Classes:', list(iris.target_names))
print('Missing values:', X.isna().sum().to_dict())
display(X.head())

## Holdout split

Set aside 20% of the examples **before** hyperparameter selection. Stratification preserves class proportions. Do not inspect the test results to choose model settings.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print('Train:', len(y_train), 'Test:', len(y_test))

## Cross-validated model selection

Scaling occurs inside each cross-validation fold through a Pipeline. The final held-out test set is not used in GridSearchCV.

In [ ]:
experiments = {
    'KNN': (
        Pipeline([('scaler', StandardScaler()), ('model', KNeighborsClassifier())]),
        {'model__n_neighbors': [3, 5, 7, 9], 'model__weights': ['uniform', 'distance']}
    ),
    'SVC': (
        Pipeline([('scaler', StandardScaler()), ('model', SVC())]),
        {'model__C': [0.1, 1, 10], 'model__kernel': ['linear', 'rbf']}
    ),
}
searches = {}
for name, (pipeline, grid) in experiments.items():
    search = GridSearchCV(pipeline, grid, cv=cv, scoring='f1_macro', n_jobs=-1)
    search.fit(X_train, y_train)
    searches[name] = search
    print(f'{name}: CV macro-F1={search.best_score_:.3f}; parameters={search.best_params_}')

## Final evaluation

Report the test results for both pre-specified model families. Accuracy, macro F1 and confusion matrices are based on the untouched holdout set. With only 30 test examples, treat estimates as uncertain.

In [ ]:
from sklearn.metrics import f1_score
rows = []
for name, search in searches.items():
    predictions = search.predict(X_test)
    rows.append({
        'model': name,
        'cv_macro_f1': search.best_score_,
        'test_accuracy': accuracy_score(y_test, predictions),
        'test_macro_f1': f1_score(y_test, predictions, average='macro'),
    })
    print(f'\n{name}\n')
    print(classification_report(y_test, predictions, target_names=iris.target_names, zero_division=0))
    ConfusionMatrixDisplay.from_predictions(
        y_test, predictions, display_labels=iris.target_names, cmap='Blues'
    )
    plt.title(f'{name}: held-out test confusion matrix')
    plt.tight_layout()
    plt.show()
results = pd.DataFrame(rows).sort_values('cv_macro_f1', ascending=False)
display(results)

## Interpretation and limitations

- Model selection uses **training-only** 5-fold stratified cross-validation.
- Both model families are evaluated on the same holdout set for a transparent comparison. Do not tune further based on the holdout results.
- This small, standard educational dataset is not evidence of real-world deployment performance.
- Previous internship notebook accuracy figures must not be described as independently validated test scores.
- To run: install `scikit-learn pandas numpy matplotlib` and execute cells in order.